# Gold Layer: Product Dimension (SCD Type 2)

**Pattern**: Slowly Changing Dimension Type 2  
**Tracks**: Full history of product attribute changes

**Source Tables**:
- `silver.crm_products_cdc` (primary product info)
- `silver.erp_product_category_cdc` (category, subcategory, maintenance flag)

**SCD Type 2 Columns**:
- `effective_from`: When this version became active
- `effective_to`: When this version was superseded (9999-12-31 for current)
- `is_current`: TRUE for active version, FALSE for historical

**Business Use Cases**:
- Track product category changes (Electronics → Clearance)
- Analyze sales with product's category at sale time
- Report on product evolution over time (price changes, rebranding)

**Example**: Product recategorized
```
product_key | product_id | category    | effective_from | effective_to  | is_current
1           | P456       | Electronics | 2023-01-01     | 2024-03-15    | FALSE
2           | P456       | Clearance   | 2024-03-15     | 9999-12-31    | TRUE
```

In [0]:
%sql
-- Build source data with SCD Type 2 columns
-- Apply deduplication using window functions (Retail Analytics Best Practice #3)
CREATE OR REPLACE TEMPORARY VIEW product_source AS
WITH ranked_products AS (
  SELECT 
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance_flag,
    pn.product_line,
    pn.start_date,
    pn.ingestion_timestamp,
    
    -- Add hash for change detection (all attributes except keys)
    md5(concat_ws('|', 
      pn.product_name,
      pn.category_id,
      pc.category,
      pc.subcategory,
      cast(pc.maintenance_flag as string),
      pn.product_line,
      cast(pn.start_date as string)
    )) as attribute_hash,
    
    -- Deduplicate: keep most recent record per product_number
    ROW_NUMBER() OVER (PARTITION BY pn.product_number ORDER BY pn.ingestion_timestamp DESC) as rn
    
  FROM workspace.silver.crm_products_cdc pn
  LEFT JOIN workspace.silver.erp_product_category_cdc pc
    ON pn.category_id = pc.category_id
  WHERE pn.end_date IS NULL -- filter out ended products, keep active ones
)
SELECT 
  product_id,
  product_number,
  product_name,
  category_id,
  category,
  subcategory,
  maintenance_flag,
  product_line,
  start_date,
  attribute_hash,
  current_date() as effective_from,
  date('9999-12-31') as effective_to,
  TRUE as is_current
FROM ranked_products
WHERE rn = 1; -- Keep only the most recent version

SELECT COUNT(*) as source_products FROM product_source;

In [0]:
%sql
-- Preview source data with SCD columns
SELECT * FROM product_source LIMIT 5;

# Writing Gold Table

## Two-Step SCD Type 2 Workflow

**Cell 5 - Initial Load** (Run once, safe to re-run):
- Creates table if it doesn't exist
- Loads all products with `is_current = TRUE`
- **Idempotent**: Won't insert duplicates if table already has data

**Cell 6 - Incremental Updates** (Run for all future updates):
- Closes out changed products (sets `effective_to = today`, `is_current = FALSE`)
- Inserts new versions for changed products
- Inserts brand new products
- **Preserves all history** - never deletes records

## Typical Usage
1. **First time**: Run Cell 2 → Cell 5
2. **After CDC updates**: Run Cell 2 → Cell 6
3. **Verify history**: Run Cell 8

In [0]:
%sql
-- Initial load: Create table and load if it doesn't exist
-- Safe to run multiple times (idempotent)

CREATE TABLE IF NOT EXISTS workspace.gold.dim_products (
  product_key BIGINT,
  product_id STRING,
  product_number STRING,
  product_name STRING,
  category_id STRING,
  category STRING,
  subcategory STRING,
  maintenance_flag STRING,
  product_line STRING,
  start_date DATE,
  attribute_hash STRING,
  effective_from DATE,
  effective_to DATE,
  is_current BOOLEAN
)
USING DELTA;

-- Only insert if table is empty (initial load)
INSERT INTO workspace.gold.dim_products
SELECT 
  ROW_NUMBER() OVER (ORDER BY product_number) as product_key,
  product_id,
  product_number,
  product_name,
  category_id,
  category,
  subcategory,
  maintenance_flag,
  product_line,
  start_date,
  attribute_hash,
  effective_from,
  effective_to,
  is_current
FROM product_source
WHERE NOT EXISTS (SELECT 1 FROM workspace.gold.dim_products LIMIT 1);

-- Show results
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT product_number) as unique_products,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions
FROM workspace.gold.dim_products;

In [0]:
%sql
-- Run this cell for incremental updates (after initial load)
-- SCD Type 2: Close out changed products and insert new versions

-- Step 1: Close out products whose attributes have changed
MERGE INTO workspace.gold.dim_products as target
USING product_source as source
ON target.product_number = source.product_number 
   AND target.is_current = TRUE
WHEN MATCHED AND target.attribute_hash <> source.attribute_hash
THEN UPDATE SET
  effective_to = current_date(),
  is_current = FALSE;

-- Step 2: Insert new versions for changed products
INSERT INTO workspace.gold.dim_products
SELECT 
  (SELECT COALESCE(MAX(product_key), 0) FROM workspace.gold.dim_products) + ROW_NUMBER() OVER (ORDER BY source.product_number) as product_key,
  source.product_id,
  source.product_number,
  source.product_name,
  source.category_id,
  source.category,
  source.subcategory,
  source.maintenance_flag,
  source.product_line,
  source.start_date,
  source.attribute_hash,
  current_date() as effective_from,
  date('9999-12-31') as effective_to,
  TRUE as is_current
FROM product_source source
INNER JOIN (
  SELECT product_number 
  FROM workspace.gold.dim_products
  WHERE effective_to = current_date()
) changed
ON source.product_number = changed.product_number;

-- Step 3: Insert brand new products (not in target at all)
INSERT INTO workspace.gold.dim_products
SELECT 
  (SELECT COALESCE(MAX(product_key), 0) FROM workspace.gold.dim_products) + ROW_NUMBER() OVER (ORDER BY source.product_number) as product_key,
  source.product_id,
  source.product_number,
  source.product_name,
  source.category_id,
  source.category,
  source.subcategory,
  source.maintenance_flag,
  source.product_line,
  source.start_date,
  source.attribute_hash,
  current_date() as effective_from,
  date('9999-12-31') as effective_to,
  TRUE as is_current
FROM product_source source
WHERE NOT EXISTS (
  SELECT 1 FROM workspace.gold.dim_products target
  WHERE target.product_number = source.product_number
);

-- Show summary
SELECT 
  'Updated' as operation,
  COUNT(*) as products
FROM workspace.gold.dim_products
WHERE effective_to = current_date()
UNION ALL
SELECT 
  'Current Total',
  COUNT(DISTINCT product_number)
FROM workspace.gold.dim_products;

## Sanity checks of gold table

In [0]:
%sql
-- Verify SCD Type 2 implementation
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT product_number) as unique_products,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_versions,
  SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) as historical_versions,
  COUNT(*) - COUNT(DISTINCT product_number) as products_with_history
FROM workspace.gold.dim_products;

-- Show products with multiple versions (history tracking)
SELECT 
  product_number,
  product_key,
  product_name,
  category,
  subcategory,
  effective_from,
  effective_to,
  is_current
FROM workspace.gold.dim_products
WHERE product_number IN (
  SELECT product_number 
  FROM workspace.gold.dim_products
  GROUP BY product_number
  HAVING COUNT(*) > 1
)
ORDER BY product_number, effective_from
LIMIT 20;